# Blind Review of the Task-Type and Handling Rules

200 cases from `derived/task-review-sample.csv` (OSHA ITA case detail, 2025): 150 drawn at random from handling cases (75 hospital, 75 nursing facility) and 50 further cases the rule classified as lateral transfers (oversampled because lateral transfers are rare and their count is the figure most likely to be cited). The reviewer (the author) labeled each case without seeing the rule's output or knowing which cases were oversampled: (1) is it a handling injury, and (2) if so, which task was under way. Labels: `derived/task-review-labels.csv`.

The rule scored here is the final rule in `handling-task-types.ipynb`, applied to the same narratives. Reads only published files; no download needed.

The reviewed sample was drawn by Section 5 of `handling-task-types.ipynb` before the lateral rule's final revision (wheelchair-to-table moved to seated transfers). Its 150 random cases are identical to those the final notebook draws; its 50 oversampled cases were selected by the earlier version of the lateral rule. The final rule is scored here on the reviewed cases.

In [1]:
import re, numpy as np, pandas as pd
s = pd.read_csv('derived/task-review-sample.csv', dtype=str)
l = pd.read_csv('derived/task-review-labels.csv', dtype=str).fillna('')
m = s.merge(l, on='row'); print(len(m), 'cases'); m['sample'].value_counts()

200 cases


sample
random                150
lateral oversample     50
Name: count, dtype: int64

In [2]:
TASKS = [
 ('lateral transfer between surfaces',
  r"\b(?:lateral(?:ly)? (?:transfer\w*|slid\w*|mov\w*)|bed to bed|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:operating|or|surgical|procedure|exam|examination|ct|cat scan|mri|x-?ray|imaging|radiology|cath\w*|ir|endoscopy|dialysis) (?:room )?(?:table|bed)|"
  r"(?:operating|or|surgical|procedure|exam|ct|mri|x-?ray|imaging) table to|"
  r"(?:bed|table|stretcher|gurney|cart|litter|cot) to (?:a |an |the |another |new |hospital |ems |shower |her |his )?(?:bed|table|stretcher|gurney|cart|litter|cot)|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:stretcher|gurney|litter|cot)\b|(?:stretcher|gurney|litter|cot) to|"
  r"slid\w* (?:the )?(?:resident|patient|pt|him|her) (?:over|across|onto|from))\b"),
 ('recovery from the floor',
  r"\b(?:off (?:of )?the (?:floor|ground)|from the (?:floor|ground)|up off|(?:found|was|lying) on the floor|lift\w* (?:\w+ ){0,2}(?:from|off) (?:the )?floor)\b"),
 ('catching or supporting a falling person',
  r"\b(?:catch\w*|caught (?:the |a )?(?:resident|patient|him|her)|began to fall|started to fall|was falling|"
  r"prevent\w* (?:\w+ ){0,2}from falling|keep \w+ from falling|lower\w* (?:\w+ ){0,2}to the (?:floor|ground)|(?:legs|knees) (?:gave out|buckled))\b"),
 ('seated or standing transfer',
  r"\b(?:(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair) to (?:a |the )?(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair)|"
  r"sit[- ]to[- ]stand|stand[- ]pivot|pivot\w*|into (?:the |a |her |his )?(?:wheel ?chair|chair|recliner|w/?c)|out of (?:the |a |her |his )?(?:wheel ?chair|chair|bed|recliner)|"
  r"on(?:to)? (?:the )?(?:toilet|commode)|off (?:the )?(?:toilet|commode)|(?:to|help\w* \w+) stand)\b"),
 ('repositioning in bed',
  r"\b(?:pull\w* (?:\w+ ){0,3}up (?:in|to)|boost\w*|reposition\w*|turn\w* (?:\w+ ){0,3}(?:in bed|over|on (?:her|his|their|the) side)|"
  r"roll\w* (?:the |a )?(?:resident|patient|pt|him|her)|draw ?sheet|turning sheet|up in bed|head of (?:the )?bed)\b"),
 ('ambulation', r"\b(?:ambulat\w*|walking (?:the |a |with (?:the |a )?)?(?:resident|patient|pt))\b"),
 ('transfer, type not stated', r"\btransfer\w*\b"),
]
SEATED_TO_STRETCHER = r"\b(?:wheel ?chair|chair|w/?c|recliner|geri ?chair) to (?:a |an |the )?(?:\w+ )?(?:stretcher|gurney|cot|table)\b"
def task(s):
    if re.search(SEATED_TO_STRETCHER, s): return 'seated or standing transfer'
    for name, p in TASKS:
        if re.search(p, s): return name
    return 'handling, task not stated'

SHORT = {'lateral transfer between surfaces': 'lateral', 'recovery from the floor': 'floor',
         'catching or supporting a falling person': 'catching', 'seated or standing transfer': 'seated',
         'repositioning in bed': 'repositioning', 'ambulation': 'ambulation',
         'transfer, type not stated': 'transfer_unstated', 'handling, task not stated': 'unstated'}
m['rule_task'] = m.narr.fillna('').str.lower().map(task).map(SHORT)
def wilson(k, n, z=1.96):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return round(p, 3), round(c - h, 3), round(c + h, 3)

## 1. Handling rule: share of rule-flagged cases that are handling (random sample)

In [3]:
rnd = m[m['sample'] == 'random']; rows = []
for st, d in rnd.groupby('setting'):
    k = int((d.human_handling == 'yes').sum()); p, lo, hi = wilson(k, len(d))
    rows.append({'setting': st, 'reviewed': len(d), 'handling': k, 'precision': p, 'ci95_low': lo, 'ci95_high': hi})
handling = pd.DataFrame(rows); handling.to_csv('derived/review-handling-precision.csv', index=False); handling

,setting,reviewed,handling,precision,ci95_low,ci95_high
0,hospital,75,60,0.800,0.696,0.875
1,nursing facility,75,67,0.893,0.803,0.945


## 2. Lateral-transfer rule: precision (all cases the rule calls lateral) and what the misclassified cases were

In [4]:
rows = []
for st, d in m[m.rule_task == 'lateral'].groupby('setting'):
    k = int((d.human_task == 'lateral').sum()); p, lo, hi = wilson(k, len(d))
    rows.append({'setting': st, 'rule lateral': len(d), 'reviewer lateral': k, 'precision': p, 'ci95_low': lo, 'ci95_high': hi})
lateral = pd.DataFrame(rows); lateral.to_csv('derived/review-lateral-precision.csv', index=False)
print(lateral.to_string(index=False))
print('\nreviewer labels for cases the rule calls lateral:')
m[m.rule_task == 'lateral'].human_task.replace('', 'not handling').value_counts()

         setting  rule lateral  reviewer lateral  precision  ci95_low  ci95_high
        hospital            54                45      0.833     0.713       0.91
nursing facility             1                 1      1.000     0.207       1.00

reviewer labels for cases the rule calls lateral:


human_task
lateral         46
seated           7
catching         1
not handling     1
Name: count, dtype: int64

## 3. Reviewer's task labels among true handling cases (random sample)

In [5]:
tk = pd.crosstab(rnd[rnd.human_handling == 'yes'].human_task, rnd[rnd.human_handling == 'yes'].setting, margins=True, margins_name='total')
tk.to_csv('derived/review-task-distribution.csv'); tk

setting,hospital,nursing facility,total
human_task,,,
catching,6,6,12
floor,0,7,7
lateral,7,2,9
repositioning,27,21,48
seated,10,17,27
transfer_unstated,9,14,23
unstated,1,0,1
total,60,67,127


## 4. Rule task vs reviewer task (random sample, true handling cases)

In [6]:
y = rnd[rnd.human_handling == 'yes']
ct = pd.crosstab(y.rule_task, y.human_task, rownames=['rule'], colnames=['reviewer']); ct.to_csv('derived/review-rule-vs-reviewer.csv'); ct

reviewer,catching,floor,lateral,repositioning,seated,transfer_unstated,unstated
rule,,,,,,,
catching,8,0,0,0,2,0,0
floor,0,5,0,1,0,0,0
lateral,0,0,7,0,0,0,0
repositioning,1,0,0,26,0,0,0
seated,2,1,1,0,12,0,0
transfer_unstated,0,0,1,0,12,11,0
unstated,1,1,0,21,1,12,1
